# Classificação de Texto com LLM
## Usando Claude Sonnet 4.6 via Databricks

Os notebooks anteriores usaram **modelos de Machine Learning clássicos** (Regressão Logística, Decision Tree, Random Forest, XGBoost) — todos treinados com features **numéricas**.

Mas e quando os dados são **texto livre**? Reviews de clientes, e-mails de suporte, comentários em redes sociais... Para isso, podemos:

1. Transformar o texto em features numéricas (TF-IDF, embeddings) e usar um classificador clássico.
2. **Usar um Large Language Model (LLM) diretamente** — sem treinar nada.

Neste notebook, vamos pelo caminho 2: vamos pedir para o **Claude Sonnet 4.6** classificar mensagens de clientes em três categorias, usando o **client da Databricks** para acessar o modelo.

### Vantagens do approach com LLM
- **Sem dados de treino**: o modelo já entende linguagem natural.
- **Flexibilidade**: mudar de tarefa = mudar o prompt.
- **Multilíngue**: funciona em PT-BR sem ajustes.

### Desvantagens
- **Custo**: cada chamada à API tem um preço.
- **Latência**: cada predição leva ~1 segundo (vs microssegundos de um modelo clássico).
- **Menos controle**: difícil garantir formato exato da saída.

## 1. Pré-requisitos

Para rodar este notebook, você precisa:

1. **Conta Databricks** com acesso a Foundation Model APIs (Claude Sonnet 4.6 disponível como serving endpoint).
2. **Personal Access Token (PAT)** do seu workspace Databricks.
3. **URL do workspace** (ex.: `https://dbc-xxxxxxxx-xxxx.cloud.databricks.com`).
4. Pacote `databricks-sdk` instalado:

```bash
pip install databricks-sdk
```

## 2. Importação das Bibliotecas

In [ ]:
import pandas as pd
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

## 3. Configuração do Cliente Databricks

> **⚠️ Substitua os valores abaixo pelos seus.**
> Não compartilhe seu token! Em produção, use variáveis de ambiente ou o Databricks Secrets.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  PREENCHA COM SUAS CREDENCIAIS                                   ║
# ╚══════════════════════════════════════════════════════════════════╝
DATABRICKS_HOST  = 'https://<seu-workspace>.cloud.databricks.com'
DATABRICKS_TOKEN = '<SEU_PERSONAL_ACCESS_TOKEN>'

# Endpoint do modelo (nome padrão do Claude Sonnet 4.6 no Databricks)
ENDPOINT_NAME = 'databricks-claude-sonnet-4-6'

# Inicializa o cliente
client = WorkspaceClient(host=DATABRICKS_HOST, token=DATABRICKS_TOKEN)
print('Cliente Databricks configurado.')

## 4. O Dataset

Vamos criar um pequeno dataset de **10 mensagens de clientes** que precisam ser classificadas em 3 categorias:

- **`duvida`** — cliente fazendo uma pergunta.
- **`reclamacao`** — cliente reportando um problema ou insatisfação.
- **`elogio`** — cliente expressando satisfação.

In [ ]:
dados = pd.DataFrame({
    'texto': [
        'Boa tarde, gostaria de saber se vocês entregam no Espírito Santo.',
        'Já se passaram 15 dias e meu pedido ainda não chegou. Inaceitável!',
        'Adorei o produto, superou minhas expectativas! Muito obrigado.',
        'O sistema está fora do ar há horas, ninguém me responde.',
        'Como faço para alterar a senha da minha conta?',
        'Atendimento maravilhoso, fui muito bem tratado pela equipe.',
        'Estou cobrando duas vezes na fatura, quero estorno imediato.',
        'Vocês têm previsão de quando o item vai voltar ao estoque?',
        'Recomendo a todos, qualidade excelente e entrega rápida!',
        'O aplicativo trava toda vez que tento finalizar a compra.',
    ],
    'classe_real': [
        'duvida', 'reclamacao', 'elogio', 'reclamacao', 'duvida',
        'elogio', 'reclamacao', 'duvida', 'elogio', 'reclamacao',
    ],
})

print(f'Dataset: {len(dados)} mensagens em 3 classes')
print(f'Distribuição: {dados["classe_real"].value_counts().to_dict()}\n')
dados

## 5. Construindo o Prompt

O segredo de uma boa classificação com LLM está no **prompt**. Algumas práticas:

1. **System prompt** define o "papel" do modelo (classificador especialista).
2. **Lista as classes** explicitamente.
3. **Pede saída em formato fixo** (uma palavra, JSON, etc.) para facilitar o parse.
4. **Temperature = 0** para resultados determinísticos.

In [ ]:
SYSTEM_PROMPT = '''Você é um classificador de mensagens de clientes de um e-commerce.
Sua tarefa é classificar cada mensagem em EXATAMENTE UMA destas três categorias:

- duvida: o cliente está fazendo uma pergunta ou pedindo informação.
- reclamacao: o cliente está relatando um problema, falha ou insatisfação.
- elogio: o cliente está expressando satisfação ou agradecimento.

Responda APENAS com a palavra da categoria, em minúsculas, sem aspas, sem pontuação, sem explicações.'''

print(SYSTEM_PROMPT)

## 6. Função de Classificação

Esta função encapsula a chamada ao modelo. Recebe um texto e devolve a classe prevista.

In [ ]:
def classificar_texto(texto: str) -> str:
    """Classifica uma mensagem usando Claude Sonnet 4.6 via Databricks."""
    response = client.serving_endpoints.query(
        name=ENDPOINT_NAME,
        messages=[
            ChatMessage(role=ChatMessageRole.SYSTEM, content=SYSTEM_PROMPT),
            ChatMessage(role=ChatMessageRole.USER, content=texto),
        ],
        max_tokens=10,
        temperature=0.0,
    )
    resposta = response.choices[0].message.content.strip().lower()

    # Normalização: garante que a resposta esteja entre as classes válidas
    classes_validas = {'duvida', 'reclamacao', 'elogio'}
    for c in classes_validas:
        if c in resposta:
            return c
    return 'desconhecido'

## 7. Teste com um Único Exemplo

Antes de rodar no dataset todo, vamos testar com uma mensagem para garantir que tudo está funcionando.

In [ ]:
exemplo = 'Vocês aceitam pagamento via PIX?'
resultado = classificar_texto(exemplo)
print(f'Texto:    {exemplo}')
print(f'Previsto: {resultado}')

## 8. Classificando o Dataset Inteiro

Agora aplicamos a função a todas as mensagens. Cada chamada vai à API do Databricks, então leva alguns segundos.

In [ ]:
previsoes = []
for i, texto in enumerate(dados['texto']):
    pred = classificar_texto(texto)
    previsoes.append(pred)
    print(f'[{i+1:2d}/{len(dados)}] real={dados["classe_real"].iloc[i]:<11s} | '
          f'previsto={pred:<11s} | {texto[:60]}...')

dados['classe_prevista'] = previsoes

## 9. Avaliação

Usamos as mesmas métricas dos notebooks anteriores — afinal, é também um problema de classificação multiclasse.

In [ ]:
acc = accuracy_score(dados['classe_real'], dados['classe_prevista'])
print(f'Acurácia: {acc:.2%} ({(dados["classe_real"] == dados["classe_prevista"]).sum()} '
      f'de {len(dados)} corretas)\n')

print(classification_report(dados['classe_real'], dados['classe_prevista']))

### 9.1 Matriz de Confusão

In [ ]:
classes = ['duvida', 'reclamacao', 'elogio']
cm = confusion_matrix(dados['classe_real'], dados['classe_prevista'], labels=classes)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes,
            cbar=False, annot_kws={'size': 14})
plt.xlabel('Previsto', fontsize=11)
plt.ylabel('Real', fontsize=11)
plt.title('Matriz de Confusão — Classificação por LLM', fontsize=13)
plt.tight_layout()
plt.show()

### 9.2 Tabela com Acertos e Erros

In [ ]:
dados['correto'] = dados['classe_real'] == dados['classe_prevista']
dados[['texto', 'classe_real', 'classe_prevista', 'correto']]

## 10. Classificando Novas Mensagens

Com a função pronta, classificar texto novo é trivial — sem retreinar nada.

In [ ]:
novas_mensagens = [
    'Quanto custa o frete para Recife?',
    'Que loja maravilhosa, vou comprar de novo!',
    'Recebi o produto quebrado, exijo a troca.',
    'O cupom de desconto não está funcionando.',
]

for msg in novas_mensagens:
    pred = classificar_texto(msg)
    print(f'[{pred:<11s}]  {msg}')

## 11. Discussão — LLM vs Modelos Clássicos

| Aspecto              | LLM (Claude)                     | Modelo Clássico (ex.: XGBoost)  |
|----------------------|----------------------------------|--------------------------------|
| **Dados de treino**  | Zero (zero-shot)                 | Centenas/milhares de exemplos  |
| **Setup**            | Prompt em texto                  | Pipeline de pré-processamento  |
| **Tipo de entrada**  | Texto livre                      | Features numéricas             |
| **Custo por predição** | Alto (chamada API)             | Baixíssimo (CPU local)         |
| **Latência**         | Centenas de ms a segundos        | Microssegundos                 |
| **Escalabilidade**   | Limitada por rate limits         | Praticamente ilimitada         |
| **Interpretabilidade** | Pode pedir explicação ao LLM   | Coeficientes / feature importance |
| **Atualização**      | Trocar prompt                    | Retreinar com novos dados      |

### Quando usar LLM para classificação?

- ✅ Você tem **pouco ou nenhum dado rotulado**.
- ✅ A tarefa envolve **compreensão semântica complexa** de texto.
- ✅ As classes podem **mudar com frequência**.
- ✅ Volume **baixo a médio** de chamadas.

### Quando preferir um modelo clássico?

- ✅ **Volume altíssimo** de predições (milhares por segundo).
- ✅ Você tem **muitos dados rotulados**.
- ✅ **Custo** e **latência** são críticos.
- ✅ **Reprodutibilidade total** é necessária (regulações).

### Híbrido — o melhor dos dois mundos
Use o LLM para **rotular um conjunto inicial** de dados, depois treine um modelo clássico em cima. Você economiza esforço humano e ganha eficiência em produção.

---

## Conclusão

Vimos como integrar **Claude Sonnet 4.6** ao fluxo de classificação usando o **client da Databricks** (`databricks-sdk`).

Os pontos centrais foram:

1. Configurar a `WorkspaceClient` com host e PAT.
2. Construir um **system prompt** claro listando as classes.
3. Pedir saída **em formato fixo** com `temperature=0.0`.
4. Avaliar com as mesmas métricas (acurácia, F1, matriz de confusão).

Esse padrão é o ponto de partida para muitas aplicações reais: triagem automática de tickets, moderação de conteúdo, classificação de feedback, roteamento de e-mails, etc.

### Próximos passos
- Adicionar **few-shot examples** ao prompt para melhorar a precisão.
- Pedir ao modelo um **JSON estruturado** com classe + confiança + justificativa.
- Implementar **retry/timeout** para robustez em produção.
- Comparar contra um modelo clássico (TF-IDF + LogisticRegression) com mais dados.